In [ ]:
%pip install datasets --quiet

In [ ]:
import logging
import os
import sys
import time
from io import StringIO

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

In [ ]:
from kubernetes import client as k8s, config as k8s_config

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN environment variables are required")
PVC_NAME = os.getenv("SHARED_PVC_NAME", "")

if not PVC_NAME:
   raise RuntimeError("SHARED_PVC_NAME environment variable is required")

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)

PVC_MOUNT_PATH = "/opt/app-root/src"

In [ ]:
import os
import gzip
import shutil
import time
import socket
import json

try:
    import s3fs
    HAS_S3FS = True
except ImportError:
    HAS_S3FS = False

socket.setdefaulttimeout(10)

PVC_NOTEBOOK_PATH = "/opt/app-root/src/"
DATASET_ROOT_NOTEBOOK = PVC_NOTEBOOK_PATH
MODEL_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "Qwen", "Qwen3-4B")
os.makedirs(MODEL_DIR, exist_ok=True)

s3_endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
s3_access_key = os.getenv("AWS_ACCESS_KEY_ID", "")
s3_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "")
s3_bucket = os.getenv("AWS_STORAGE_BUCKET", "")
s3_prefix = os.getenv("AWS_STORAGE_BUCKET_GRPO_DIR", "")

data_download_successful = False

if HAS_S3FS and s3_endpoint and s3_bucket:
    try:
        endpoint_url = (
            s3_endpoint
            if s3_endpoint.startswith("http")
            else f"https://{s3_endpoint}"
        )
        prefix = (s3_prefix or "").strip("/")

        print(
            f"[notebook] S3 configured: "
            f"endpoint={endpoint_url}, bucket={s3_bucket}, prefix={prefix or '<root>'}"
        )

        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        remote_path = f"{s3_bucket}/{prefix}" if prefix else s3_bucket
        pulled_any = False
        file_count = 0

        print(f"[notebook] Starting S3 download from prefix: {prefix}")
        for remote_file in fs.find(remote_path):
            file_count += 1

            if remote_file.endswith("/"):
                continue

            rel = remote_file[len(remote_path):].lstrip("/")
            if not rel:
                continue
            print(f"[notebook] Processing rel={rel}")

            if "qwen" in rel.lower() or (prefix and any(rel.endswith(ext) for ext in [".bin", ".json", ".model", ".safetensors", ".txt"])):
                dst = os.path.join(MODEL_DIR, rel.split("Qwen3-4B/")[-1] if "Qwen3-4B" in rel else os.path.basename(rel))
                print(f"[notebook] Routing to model dir: {dst}")
            else:
                dst = os.path.join(DATASET_ROOT_NOTEBOOK, rel)
                print(f"[notebook] Routing to default dir: {dst}")

            os.makedirs(os.path.dirname(dst), exist_ok=True)

            if not os.path.exists(dst):
                print(f"[notebook] Downloading s3://{remote_file} -> {dst}")
                t0 = time.time()
                fs.get(remote_file, dst)
                print(f"[notebook] DONE in {time.time() - t0:.2f}s")
                pulled_any = True
            else:
                print(f"[notebook] Skipping existing file {dst}")
                pulled_any = True

            if dst.endswith(".gz") and os.path.exists(dst):
                out_path = os.path.splitext(dst)[0]
                if not os.path.exists(out_path):
                    print(f"[notebook] Decompressing {dst} -> {out_path}")
                    try:
                        with gzip.open(dst, "rb") as f_in, open(out_path, "wb") as f_out:
                            shutil.copyfileobj(f_in, f_out)
                    except Exception as e:
                        print(f"[notebook] Failed to decompress {dst}: {e}")
                    else:
                        try:
                            os.remove(dst)
                        except Exception:
                            pass

        if pulled_any:
            if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
                print(f"[notebook] S3 download successful. Processed {file_count} files")
                data_download_successful = True
            else:
                print(f"[notebook] S3 downloaded {file_count} files but model not found, will try HuggingFace fallback")
        else:
            print(f"[notebook] S3 download found no files to download")

    except Exception as e:
        print(f"[notebook] S3 fetch failed: {e}")
        import traceback
        traceback.print_exc()
        print("[notebook] Will attempt HuggingFace fallback...")
else:
    if not HAS_S3FS:
        print("[notebook] S3 not available: s3fs not installed")
    else:
        print("[notebook] S3 not configured (missing endpoint or bucket env vars)")

In [ ]:
from huggingface_hub import snapshot_download

hf_token = os.getenv("HUGGINGFACE_HUB_TOKEN")

if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    print(f"Using local model from S3: {MODEL_DIR}")
else:
    print("[notebook] Model not found in S3, downloading from HuggingFace...")
    snapshot_download(
        repo_id="Qwen/Qwen3-4B",
        local_dir=MODEL_DIR,
        token=hf_token,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    print(f"Model downloaded to: {MODEL_DIR}")

In [ ]:
LOCAL_MODEL_PATH = "/opt/app-root/src/Qwen/Qwen3-4B"

# GRPO uses Agent-Ark/Toucan-1.5M (HuggingFace dataset downloaded at training time).
# For the E2E test we use minimal iterations to keep wall-clock time reasonable.
DATA_PATH = "Agent-Ark/Toucan-1.5M"
DATA_CONFIG = "Qwen3"

params = {
    # Model and data
    'model_path': LOCAL_MODEL_PATH,
    'data_path': DATA_PATH,
    'data_config': DATA_CONFIG,
    'ckpt_output_dir': '/opt/app-root/src/grpo-output',
    'backend': 'art',

    # GRPO hyperparameters — small values for fast E2E validation
    'num_iterations': 2,
    'group_size': 4,
    'prompt_batch_size': 10,
    'n_train': 50,
    'learning_rate': 1e-5,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 8,

    # vLLM GPU memory sharing
    'gpu_memory_utilization': 0.45,

    # Single node (overridden by NNODES env var if set)
    'nnodes': int(os.getenv("NNODES", "1")),
}

In [ ]:
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms
from kubeflow.trainer.rhai import TrainingHubTrainer
from kubeflow.common.types import KubernetesBackendConfig

backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_cfg)

In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME")

if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

th_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == training_runtime_name:
        th_runtime = runtime
        print("Found runtime: " + str(th_runtime))
        break

if th_runtime is None:
    raise RuntimeError(f"Required runtime '{training_runtime_name}' not found")

In [ ]:
from kubeflow.trainer.options.kubernetes import (
    PodTemplateOverrides,
    PodTemplateOverride,
    PodSpecOverride,
    ContainerOverride,
)

cache_root = "/opt/app-root/src/.cache/huggingface"
triton_cache = "/tmp/.triton"

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.LORA_GRPO,
        func_args=params,
        env={
            "HF_HOME": cache_root,
            "TRITON_CACHE_DIR": triton_cache,
            "XDG_CACHE_HOME": "/opt/app-root/src/.cache",
            "NCCL_DEBUG": "INFO",
            "TRANSFORMERS_ATTN_BACKEND": "sdpa",
        },
        resources_per_node={
            "cpu": 8,
            "memory": "64Gi",
            "nvidia.com/gpu": 1
        },
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    volumes=[
                        {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                        {"name": "dshm", "emptyDir": {"medium": "Memory"}},
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {"name": "work", "mountPath": "/opt/app-root/src", "readOnly": False},
                                {"name": "dshm", "mountPath": "/dev/shm"},
                            ],
                        )
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)

In [ ]:
# Wait for Running, then wait for Complete or Failed.
# GRPO takes longer than SFT/LoRA due to vLLM rollout + RL training loop.
client.wait_for_job_status(name=job_name, status={"Running"}, timeout=600)
client.wait_for_job_status(name=job_name, status={"Complete", "Failed"}, timeout=3600)

job = client.get_job(name=job_name)
pod_logs = client.get_job_logs(name=job_name, follow=False)

logs = []
for log_line in pod_logs:
    logs.extend(str(log_line).splitlines())

log_text = "\n".join(logs)

print(f"Training job final status: {job.status}")

if job.status == "Failed":
    print(f"ERROR: Training job '{job_name}' has Failed status")
    print("Last 30 lines of logs:")
    for line in logs[-30:]:
        print(line)
    raise RuntimeError(f"Training job '{job_name}' failed")

if "[PY] LORA_GRPO training complete. Result=" not in log_text:
    print(f"ERROR: Training completion message not found in logs")
    print("Last 50 lines of logs:")
    for line in logs[-50:]:
        print(line)
    raise RuntimeError(f"Training did not complete successfully - missing completion message")

print(f"Training job '{job_name}' completed successfully")

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

In [ ]:
logs = client.get_job_logs(name=job_name, follow=False)

logs = list(logs)
log_text = "\n".join(str(line) for line in logs)
print(log_text)

In [ ]:
client.delete_job(job_name)